<div align="center">

# Hafta 3: Derin öğrenme ve evrişimli sinir ağları

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/lab.html) · [PDF ders notu](Hafta03_Ders_Notlari.pdf) · [Colab'da aç (yedek yol)](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb)

</div>

**Bu Jupyter defteri nasıl kullanılır?** Hücreleri yukarıdan aşağıya Shift+Enter ile çalıştırın. Bölüm 0 ortamı hazırlar ve ilk önce çalıştırılmalıdır. Alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerde durur ve ancak denedikten sonra açılmalıdır. *Gözlemle ve düşün* hücreleri açık uçlu sorular sorar, bunların çözüm hücresi yoktur. *Python notu* hücreleri, yeni bir yapının ilk kullanıldığı yerde dili açıklar. Deftere ortadan başlayan katılımcı, o bölümün başındaki *Telafi* hücresini çalıştırır.

**Üç yol.** Etiketsiz hücrelerin tümü **temel yolu** oluşturur ve hiç kod yazmamış olanlar dâhil herkes için yeterlidir. **İleri yol** etiketli hücreler, programlama bilenler için isteğe bağlı genişletmelerdir. Üçüncü yolda kod yazmak yerine bir yapay zekâ (YZ) asistanına prompt (istem) yazılır: **Prompt yolunda** her alıştırmanın altında hazır promptu içeren katlanmış bir *Prompt yolu* hücresi bulunur ve her bölüm bir *Prompt görevi* ile biter. Promptu asistana kopyalayın, alıştırma hücresinin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın; sonucun doğru olup olmadığını denetim satırı gösterir. Haftanın [prompt kütüphanesi](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md) bütün promptları bir araya getirir; [prompt yolu rehberi](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/PROMPT_PATH.tr.md) ise bu defterle çalışan asistanları anlatmaktadır.

**Nerede çalışır?** Defter, kendi bilgisayarınızdaki Anaconda içinde Jupyter için hazırlanmıştır (ders klasöründeki SETUP.tr.md dosyasına bakın). Anaconda kurulamıyorsa aynı defter, en üstteki bağlantıyla Google Colab'da açılabilir. Bu defterdeki ağların eğitimi sıradan bir dizüstü bilgisayarda hücre başına birkaç saniye ile yaklaşık bir dakika arasında sürer; ekran kartı gerekmez. Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelmektedir. Gerçek hasta verisi yalnızca kendi bilgisayarınızda ve kurumunuzun kurallarıyla işlenir; bulut tabanlı bir deftere ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

İlk hücre ders klasörünü bulur. Defter tek başına açıldıysa, örneğin Google Colab'da, `vclinic` ders paketini ve veri dosyalarını GitHub'dan indirir. İkinci hücre paketleri yükler, mesajların dilini seçer ve haftanın isteğe bağlı paketlerini denetler. PyTorch, eğitimin derin öğrenme paketidir. Colab'da hazır gelir; Anaconda için tek satırlık kurulum SETUP.tr.md dosyasında anlatılmaktadır.

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
    "data/examples/echo/VC-0821.png",
    "data/examples/echo/VC-1873.png",
    "data/examples/echo/VC-1212.png",
    "models/week03/echo_cnn.pt",
    "models/week03/echo_cnn.json",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopya, örneğin Colab'da: Paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy
import math

from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from vclinic.imaging import CMR_CLASSES, ECHO_CLASSES, echo_images, make_cmr_set

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError as error:
    raise ImportError("PyTorch kurulu değil. SETUP.tr.md dosyasında anlatıldığı gibi kurun (pip install torch), ardından çekirdeği yeniden başlatın.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 2, 0):
    raise ImportError("vclinic ders paketi bu defterden eski. Ders klasörünü güncelleyin (git pull komutuyla ya da ZIP dosyasını yeniden indirerek) ve çekirdeği yeniden başlatın.")
teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
for name in ["ipywidgets", "torchvision"]:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        print(f"{name}: kurulu değil (isteğe bağlı; defter yerleşik bir yedek yol kullanır)")

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 3.1": (49152, 0),
    "Alıştırma 3.2": (0.148, 0.001),
    "Alıştırma 3.3": (0.207, 0.001),
    "Alıştırma A": (0.859, 0.001),
    "Alıştırma B": (-0.5, 0.001),
    "Alıştırma C": (40, 0),
    "Alıştırma D": (31, 0),
    "Alıştırma E": (10098, 0),
    "Alıştırma F": (1.429, 0.001),
    "Alıştırma G": (0.916, 0.001),
    "Alıştırma H": (4640, 0),
    "Alıştırma I": (73984, 0),
    "Alıştırma J": (0.822, 0.02),
})
print("Beklenen cevaplar hazır.")

## Python adımı 3: Görüntüleri dizi olarak işlemek

Bilgisayar için gri tonlu bir görüntü bir sayı tablosudur: Her piksel için genellikle 0 (siyah) ile 1 (beyaz) arasında bir sayı. Bir görüntü yığını üç boyutlu bir dizidir; renkli bir görüntüye kanallar için dördüncü bir boyut eklenir. Bu adım elle bir görüntü oluşturur, dosyadan bir görüntü okur, ekokardiyografi karelerinden oluşan bir yığını dilimler ve normalleştirir; sonunda da tek bir nöronun eğitim döngüsünü yazar. Defterin sonraki bütün bölümleri bu adımları kullanmaktadır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
img = np.zeros((7, 7))
img[1:6, 1:6] = 1.0      # parlak bir kare
img[2:5, 2:5] = 0.2      # küçük bir ventrikül gibi koyu bir merkez
print(img.shape, img.dtype)
print(img)
plt.figure(figsize=(2.2, 2.2))
plt.imshow(img, cmap="gray", vmin=0, vmax=1)
plt.axis("off")
plt.show()

**Python notu.** `np.zeros((7, 7))` 0 ile dolu, 7 satır ve 7 sütunluk bir dizi oluşturur. `img[1:6, 1:6]` 1-5. satırları ve 1-5. sütunları seçer: Bir `a:b` dilimine ilk sayı dâhildir, ikincisi dâhil değildir. İlk indis her zaman yukarıdan sayılan satır, ikincisi sütundur. `plt.imshow` diziyi çizer; `vmin` ve `vmax` hangi değerlerin siyah ve beyaz olacağını sabitler.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
frame = plt.imread(ROOT / "data" / "examples" / "echo" / "VC-1212.png")
print(frame.shape, frame.dtype, round(float(frame.min()), 3), round(float(frame.max()), 3))
plt.figure(figsize=(3, 3))
plt.imshow(frame, cmap="gray", vmin=0, vmax=1)
plt.title("VC-1212")
plt.axis("off")
plt.show()

**Python notu.** `plt.imread` bir görüntü dosyasını diziye okur. Portable Network Graphics (PNG) biçimindeki gri tonlu bir dosya, değerleri 0 ile 1 arasında olan iki boyutlu bir dizi verir; renkli bir dosya ise (satır, sütun, 3) ya da (satır, sütun, 4) boyutunda bir dizi verirdi. `ROOT / "data" / ...` bir yolun parçalarını Windows, macOS ve Linux'ta aynı biçimde birleştirir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))
X12, y12, table12 = echo_images(registry.head(12))
print(X12.shape, y12)
print(X12[0].shape, X12[:, 32, :].shape, round(float(X12[0, 32, 32]), 3))

**Python notu.** `echo_images`, aldığı tablodaki her hasta için bir kare çizer ve üç şey döndürür; satır bunları üç ada dağıtır. `X12` üç boyutludur: kare, satır ve sütun. `X12[0]` ilk karedir; `X12[:, 32, :]` ise her karenin 32. satırını alır. Tek başına iki nokta *hepsi* anlamına gelir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
x = X12[0]
z = (x - x.mean()) / x.std()                      # z skorları: ortalama 0, SS 1
m = (x - x.min()) / (x.max() - x.min())           # en küçük-en büyük: 0'dan 1'e
print(round(float(z.mean()), 3), round(float(z.std()), 3), round(float(m.min()), 3), round(float(m.max()), 3))
crop = x[16:48, 16:48]
print("kırpma", crop.shape)

**Python notu.** Bir dizi üzerindeki aritmetik işlemler her elemana aynı anda uygulanır; böylece `x - x.mean()` ortalamayı 4096 pikselin hepsinden çıkarır. Bir ağ, 0 çevresinde ve yayılımı 1'e yakın girdilerden daha kolay öğrenir. Ortalama ve standart sapma (SS), 3. bölümde görüleceği gibi yalnızca eğitim görüntülerinden alınmalıdır. `x[16:48, 16:48]` ortadaki 32 × 32 pikseli kırpar.

In [ ]:
# Python alıştırması: tek bir nöronun eğitim döngüsü
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


feats = registry[["lvedd_mm", "ivs_mm"]].to_numpy(float)
target = ((registry["lvedd_mm"] > 58) | (registry["ivs_mm"] >= 13)).astype(int).to_numpy()
Xs = (feats - feats.mean(axis=0)) / feats.std(axis=0)

w, b, lr = np.zeros(2), 0.0, 0.5
for step in range(201):
    p = sigmoid(Xs @ w + b)                                              # 1. ileri geçiş
    loss = -np.mean(target * np.log(p) + (1 - target) * np.log(1 - p))  # 2. kayıp
    grad_w = Xs.T @ (p - target) / len(target)                           # 3. gradyanlar
    grad_b = np.mean(p - target)
    if step % 50 == 0:
        print(step, round(loss, 4), np.round(w, 3), round(b, 3))
    w = w - lr * grad_w                                                  # 4. güncelleme
    b = b - lr * grad_b

**Python notu.** Her eğitim döngüsünün aynı dört adımı vardır: *İleri geçiş* tahminleri hesaplar, *kayıp* bunların ne kadar yanlış olduğunu ölçer, *gradyanlar* kaybı azaltmak için her ağırlığın hangi yöne gitmesi gerektiğini söyler, *güncelleme* ise ağırlıkları bu yönde öğrenme hızı kadar küçük bir adım taşır. `Xs @ w`, 2000 × 2'lik tabloyu iki ağırlıkla çarpar. `for step in range(201)` girintili satırları 201 kez tekrarlar; `step % 50 == 0` her 50 adımda bir doğru olur.

**Hızlı kontrol.** `X12` dizisinin boyutu (12, 64, 64)'tür. `X12[:, 10:20, 10:20]` dizisinin boyutu nedir?

<details><summary>Cevap</summary>

(12, 10, 10): 12 karenin hepsi, 10-19. satırlar ve 10-19. sütunlar.

</details>

**Alıştırma 3.1.** `X12` toplam kaç sayı tutmaktadır?

<details><summary><b>Prompt yolu</b>, Alıştırma 3.1: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `X12`, (12, 64, 64) boyutunda bir NumPy dizisidir: 64 × 64 piksellik 12 kare, değerler 0 ile 1 arasında.
Görev: `X12` dizisinin tuttuğu bütün sayıları say.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Sayı, tam sayı olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma 3.1", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Yöntemi önce 24 sayı tutan `np.zeros((2, 3, 4))` dizisinde dene.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma 3.1", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = int(X12.size)
teach.check("Alıştırma 3.1", answer)

**Alıştırma 3.2.** VC-1212.png dosyasından okunan `frame` görüntüsündeki piksellerin ortalama değeri nedir? Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma 3.2: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `frame`, VC-1212.png dosyasından `plt.imread` ile okunmuş (64, 64) boyutunda bir NumPy dizisidir; değerler 0 ile 1 arasındadır.
Görev: Bütün piksellerinin ortalama değerini hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Ortalama, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma 3.2", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: En küçük ve en büyük değeri de yazdır; ortalama bu ikisinin arasında olmalı, en büyük değer 1'i aşmamalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma 3.2", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(float(frame.mean()), 3)
teach.check("Alıştırma 3.2", answer)

**Alıştırma 3.3.** Nöronu sıfır ağırlıklardan yeniden eğitin, ancak 0,5 yerine 0,05 öğrenme hızıyla. 200 güncellemeden sonra kayıp nedir? Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma 3.3: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `Xs`, 2000 hastanın standartlaştırılmış lvedd_mm ve ivs_mm değerlerini tutan (2000, 2) boyutunda bir NumPy dizisidir (ortalama 0, SS 1). `target`, 0 ya da 1 değerlerinden oluşan bir NumPy dizisidir (1 anormal).
Görev: Tek bir lojistik nöronu gradyan inişiyle eğit: w = [0, 0] ve b = 0'dan başla, 0,05 öğrenme hızıyla 200 güncelleme yap. Her güncelleme p = sigmoid(Xs @ w + b) değerini hesaplar, ardından Xs.T @ (p − target) / n ve mean(p − target) gradyanlarının 0,05 katını çıkarır. 200 güncellemeden sonra ortalama ikili çapraz entropiyi hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: 200 güncellemeden sonraki kayıp, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma 3.3", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Python adımındaki 0,5 öğrenme hızıyla aynı döngü, orada 200. adımda yazdırılan kaybı vermeli.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma 3.3", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
w3, b3 = np.zeros(2), 0.0
for step in range(200):
    p3 = sigmoid(Xs @ w3 + b3)
    w3 = w3 - 0.05 * Xs.T @ (p3 - target) / len(target)
    b3 = b3 - 0.05 * np.mean(p3 - target)
p3 = sigmoid(Xs @ w3 + b3)
answer = round(float(-np.mean(target * np.log(p3) + (1 - target) * np.log(1 - p3))), 3)
teach.check("Alıştırma 3.3", answer)

## 1. NumPy ile bir nöron ve bir ağ

*Ders: [Tek sayfada yapay sinir ağları](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#neural-networks-in-one-page)*

Bir nöron girdilerini her birini kendi ağırlığıyla toplar, bir sabit terim ekler ve toplamı bir *aktivasyon fonksiyonundan* geçirir. Bir katman, aynı girdileri gören nöronlardan oluşan bir sıradır; bir ağ ise katmanları üst üste dizer ve her katman bir öncekinin çıktıları üzerinde çalışır. Doğrusal olmayan bir aktivasyon olmasaydı, üst üste dizilen katmanlar tek bir düz sınıra indirgenirdi.

**Formüllerden koda.** Ağırlıklı toplam $z = \mathbf{w} \cdot \mathbf{x} + b$, `w @ x + b` ile hesaplanır. Sigmoid $\sigma(z) = 1/(1 + e^{-z})$ `sigmoid(z)`, düzeltilmiş doğrusal birim (ReLU) $\max(0, z)$ `np.maximum(0, z)` ile yazılır; softmax $e^{z_k}/\sum_j e^{z_j}$ ise son katmanın çıktılarını toplamı 1 olan olasılıklara çevirir. `W1` ağırlıkları (girdi, nöron) boyutunda olan bir katman, bütün nöronları için `x @ W1 + b1` değerini tek seferde hesaplar.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 1. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(1)

In [ ]:
# Üç aktivasyon fonksiyonu
zz = np.linspace(-5, 5, 201)
fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.plot(zz, sigmoid(zz), color="#9B1B1B", lw=2, label="sigmoid")
ax.plot(zz, np.tanh(zz), color="#1F5F8B", lw=2, label="tanh")
ax.plot(zz, np.maximum(0, zz), color="#B97813", lw=2, label="ReLU")
ax.set_ylim(-1.2, 3)
ax.axhline(0, color="#888888", lw=0.8)
ax.set_xlabel("z")
ax.legend()
plt.show()

In [ ]:
# Küçük bir ağ: 2 girdi, ReLU kullanan 3 gizli nöron, softmax kullanan 2 çıktı
def softmax(v):
    e = np.exp(v - v.max())
    return e / e.sum()


W1 = np.array([[1.5, -1.0, 0.5],
               [0.2, 1.2, -0.8]])
b1 = np.array([0.0, -0.5, 0.1])
W2 = np.array([[-1.0, 1.0],
               [-1.2, 1.2],
               [0.8, -0.8]])
b2 = np.array([0.6, -0.6])

x_case = Xs[registry["patient_id"] == "VC-1212"][0]     # vakanın standartlaştırılmış lvedd ve ivs değerleri
hidden = np.maximum(0, x_case @ W1 + b1)
out = softmax(hidden @ W2 + b2)
print("x =", np.round(x_case, 3))
print("gizli katman =", np.round(hidden, 3))
print("olasılıklar (normal, anormal) =", np.round(out, 3))

**Python notu.** `np.array([[...], [...]])` bir tabloyu satır satır yazar: `W1` 2 satır (girdiler) ve 3 sütundan (nöronlar) oluşur. `x_case @ W1` her gizli nöron için bir ağırlıklı toplam verir. `softmax` içinde en büyük değerin çıkarılması sonucu değiştirmez, ancak büyük sayılarda `np.exp` fonksiyonunun taşmasını önler. Buradaki ağırlıklar elle verilmiştir; 2. bölümde ağ onları kendisi öğrenir.

**Gözlemle ve düşün.** `np.maximum(0, ...)` yerine birim fonksiyonu, yani `x_case @ W1 + b1` ifadesini kullanın ve iki katmanın birlikte neye dönüştüğünü kâğıt üzerinde çıkarın. Bir ağ katmanları arasında neden doğrusal olmayan bir aktivasyona ihtiyaç duyar?

**Alıştırma A.** Bir nöron iki standartlaştırılmış ölçümü, `x = [1.2, -0.5]`, almaktadır. Ağırlıkları `w = [0.8, -1.5]`, sabit terimi `b = 0.1` olan nöron sigmoid kullanmaktadır. Çıktısı nedir? Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma A: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: İki standartlaştırılmış ölçüm `x = [1.2, -0.5]`, ağırlıklar `w = [0.8, -1.5]` ve sabit terim `b = 0.1`.
Görev: Ağırlıklı toplam z = w · x + b'yi ve sigmoid çıktısını, 1 / (1 + e^(−z)), hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Sigmoid çıktısı, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma A", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: w = [0, 0] ve b = 0 iken çıktı 0,5 olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma A", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
z_a = np.array([0.8, -1.5]) @ np.array([1.2, -0.5]) + 0.1
answer = round(float(sigmoid(z_a)), 3)
teach.check("Alıştırma A", answer)

**Prompt görevi, 1. bölüm: Aktivasyon fonksiyonları ve eğimleri.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: Veri yok: −5 ile 5 arasında z değerlerinden oluşan bir ızgara.
Görev: Sigmoid, tanh ve ReLU fonksiyonlarını bir panelde, türevlerini ikinci bir panelde çiz.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: İki panelli bir şekil. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Her türevin en büyük değerini yazdır.
</pre>

</details>

**Beklenen sonuç.** Sigmoidin türevi z = 0'da 0,25 ile, tanh'nin türevi 1 ile en yüksek değerine ulaşır; ReLU'nun türevi negatif z için 0, pozitif z için 1'dir.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 2. Öğrenme: PyTorch ile kayıp, gradyanlar ve eğitim döngüsü

*Ders: [Öğrenme: Kayıp, gradyanlar ve eğitim döngüsü](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#learning-loss-gradients-and-the-training-loop)*

Bir ağ, tahminleri yanlış oldukça büyüyen bir sayıyı, yani *kaybı* azaltarak öğrenir. *Geri yayılım*, analizdeki zincir kuralını kayıptan geriye doğru her ağırlığa uygular ve kaybın en hızlı arttığı yönü, yani *gradyanı* verir. *Gradyan inişi* her ağırlığı gradyanının tersi yönünde küçük bir adım taşır. PyTorch gradyanları kendisi hesaplar; buna *autograd* denir.

**Formüllerden koda.** Etiketi $y$ ve tahmin edilen olasılığı $p$ olan tek bir hasta için ikili çapraz entropi $L = -[y \ln p + (1-y) \ln(1-p)]$ ile hesaplanır: `F.binary_cross_entropy(p, y)`. Sigmoid kullanan bir nöron için gradyanı $\partial L/\partial w_j = (p - y)\,x_j$'dir. $w \leftarrow w - \eta\,\partial L/\partial w$ güncellemesini, `loss.backward()` `w.grad` değerini doldurduktan sonra `optimizer.step()` yapar. Birden çok sınıfta `F.cross_entropy(logits, labels)` softmax ile çapraz entropiyi tek adımda uygular.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 2. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(2)

In [ ]:
# Autograd gradyan formülünü doğrular
x_t = torch.tensor([1.2, -0.5])
w_t = torch.tensor([0.8, -1.5], requires_grad=True)
b_t = torch.tensor(0.1, requires_grad=True)
p_t = torch.sigmoid(w_t @ x_t + b_t)
loss_t = F.binary_cross_entropy(p_t, torch.tensor(1.0))
loss_t.backward()
print("p =", round(p_t.item(), 3), " kayıp =", round(loss_t.item(), 4))
print("autograd:", np.round(w_t.grad.numpy(), 4), round(b_t.grad.item(), 4))
print("formül (p − y) · x:", np.round((p_t.item() - 1.0) * x_t.numpy(), 4))

**Python notu.** Tensör, PyTorch'un dizisidir. `requires_grad=True`, PyTorch'tan tensörle yapılan her işlemi hatırlamasını ister; böylece `loss_t.backward()` bu işlemlerin üzerinden geriye giderek gradyanı `w_t.grad` içine yazabilir. `.item()` tek değerli bir tensörü Python sayısına, `.numpy()` ise bir tensörü NumPy dizisine çevirir.

In [ ]:
# 8 gizli nöronlu bir ağ raporu iki ölçümden öğrenir
Xs_t = torch.tensor(Xs, dtype=torch.float32)
t_t = torch.tensor(target)
torch.manual_seed(RANDOM_STATE)
mlp = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.05)
for step in range(301):
    loss = F.cross_entropy(mlp(Xs_t), t_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(step, round(loss.item(), 4))

g1, g2 = np.meshgrid(np.linspace(-3, 5, 200), np.linspace(-3, 5, 200))
grid = np.c_[g1.ravel(), g2.ravel()]
with torch.no_grad():
    p_mlp = F.softmax(mlp(torch.tensor(grid, dtype=torch.float32)), dim=1)[:, 1].numpy().reshape(g1.shape)
p_one = sigmoid(grid @ w + b).reshape(g1.shape)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for ax, pr, name in [(axes[0], p_one, "tek nöron"), (axes[1], p_mlp, "8 gizli nöron")]:
    ax.contourf(g1, g2, pr, levels=[0, 0.5, 1], colors=["#EAF1F7", "#F6E0E0"])
    ax.scatter(Xs[target == 0, 0], Xs[target == 0, 1], s=4, color="#1F5F8B", alpha=0.4, label="normal")
    ax.scatter(Xs[target == 1, 0], Xs[target == 1, 1], s=6, color="#9B1B1B", label="anormal")
    ax.set_title(name)
    ax.set_xlabel("lvedd_mm (z)")
axes[0].set_ylabel("ivs_mm (z)")
axes[1].legend(loc="upper right")
plt.show()
with torch.no_grad():
    acc_mlp = (mlp(Xs_t).argmax(1).numpy() == target).mean()
print("doğruluk: tek nöron", round(((sigmoid(Xs @ w + b) > 0.5) == target).mean(), 3),
      "| ağ", round(acc_mlp, 3))

**Python notu.** `nn.Sequential` katmanları art arda bağlar: `nn.Linear(2, 8)` her biri 2 girdi alan 8 nöronluk bir katmandır; ardından ReLU, ardından 2 çıktı nöronlu bir katman gelir. Her döngü Python adımındaki dört adımı içerir: Tek satırda ileri geçiş ve kayıp, eski gradyanları silmek için `optimizer.zero_grad()`, yenileri için `loss.backward()` ve güncelleme için `optimizer.step()`. `torch.no_grad()`, öğrenme gerekmediğinde autograd'ın kayıt tutmasını kapatır.

**Gözlemle ve düşün.** Rapor, ventrikül dilate *ya da* septum kalın olduğunda kareyi anormal sayar. Soldaki panelde tek bir nöron neden yalnızca tek bir düz sınır çizebilir de gizli katmanlı ağ sınırını köşenin çevresinden büker?

**Alıştırma B.** Bir hastanın değerleri `x = [1.0, 2.0]`, etiketi `y = 1`'dir. Sigmoid kullanan bir nöronun ağırlıkları `w = [0.5, -0.25]`, sabit terimi 0'dır. İkili çapraz entropiyi PyTorch ile hesaplayın, `backward()` çağırın ve kaybın ilk ağırlığa göre gradyanını verin. Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma B: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `x = [1.0, 2.0]` değerli ve `y = 1` etiketli tek bir hasta ile ağırlıkları `w = [0.5, -0.25]`, sabit terimi 0 olan ve sigmoid kullanan tek bir nöron. Kayıp, ikili çapraz entropidir.
Görev: w'yi `requires_grad=True` ile bir tensör yap, kaybı `F.binary_cross_entropy` ile hesapla, `backward()` çağır ve kaybın ilk ağırlığa göre gradyanını oku.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: w1'e göre gradyan, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma B", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Sonucu (p − y) · x1 formülüyle karşılaştır; iki değer eşit olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma B", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
w_b = torch.tensor([0.5, -0.25], requires_grad=True)
p_b = torch.sigmoid(w_b @ torch.tensor([1.0, 2.0]))
F.binary_cross_entropy(p_b, torch.tensor(1.0)).backward()
answer = round(float(w_b.grad[0]), 3)
teach.check("Alıştırma B", answer)

**Prompt görevi, 2. bölüm: Üç öğrenme hızı.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `Xs`, 2000 hastanın standartlaştırılmış lvedd_mm ve ivs_mm değerlerini tutan (2000, 2) boyutunda bir NumPy dizisidir (ortalama 0, SS 1). `target`, 0 ya da 1 değerlerinden oluşan bir NumPy dizisidir (1 anormal).
Görev: Python adımındaki lojistik nöronu her seferinde sıfır ağırlıklardan başlayarak 0,01, 0,1 ve 1,0 öğrenme hızlarıyla 300 güncelleme boyunca eğit; üç hız için kaybı güncelleme sayısına karşı çiz.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Üç çizgili bir şekil ve her hızın son kaybı. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Kayıplar aynı değerden, log(2) = 0,693'ten başlamalı.
</pre>

</details>

**Beklenen sonuç.** 0,01 hızı diğer ikisinin belirgin biçimde üstünde kalır; en hızlı düşüş 1,0 hızındadır. Çok küçük bir hız adımları boşa harcar.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 3. Eko laboratuvarının kareleri

*Ders: [Dizi olarak görüntüler: Eko laboratuvarının kareleri](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#images-as-arrays-the-frames-of-the-echo-laboratory)*

Sanal kliniğin eko laboratuvarında 2000 hastanın her biri için parasternal kısa eksen görüntüsünden bir diyastol sonu karesi vardır. Her karedeki ventrikül büyüklüğü ve duvar kalınlığı, hastanın kayıttaki ölçümlerinden gelir. Laboratuvarın raporu, sol ventrikül dilate (çap 58 mm'nin üstünde) ya da septum kalın (13 mm veya daha fazla) olduğunda kareyi anormal sayar. Yaklaşık her on kareden biri anormaldir.

**Formüllerden koda.** Kareler bir eğitim kümesine (yüzde 60), dönem seçimi için bir doğrulama kümesine (yüzde 20) ve sonda yalnızca bir kez kullanılan bir test kümesine (yüzde 20) ayrılır. Her kare, *eğitim* karelerinin ortalaması $\mu$ ve SS'si $\sigma$ ile $x' = (x - \mu)/\sigma$ biçiminde normalleştirilir. PyTorch görüntüleri (sayı, kanal, satır, sütun) boyutunda bekler; burada bu boyut (n, 1, 64, 64)'tür.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 3. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(3)

In [ ]:
# Eko laboratuvarının 2000 karesinin tamamı
X, y, echo_table = echo_images(registry)
print(X.shape, "anormal:", int(y.sum()), "/", len(y))
print(echo_table[["dilated", "thick_septum"]].sum().to_dict())
show = list(np.where(y == 0)[0][:4]) + list(np.where(y == 1)[0][:4])
fig, axes = plt.subplots(2, 4, figsize=(9, 4.9))
for ax, i in zip(axes.ravel(), show):
    ax.imshow(X[i], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{echo_table.patient_id[i]} ({echo_table.report[i]})\n{echo_table.lvedd_mm[i]:.0f} / {echo_table.ivs_mm[i]:.1f} mm", fontsize=8)
    ax.axis("off")
plt.show()

In [ ]:
# Eğitim, doğrulama ve test kümeleri; eğitim kareleriyle normalleştirilmiş
idx = np.arange(len(y))
train_idx, test_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
train_idx, val_idx = train_test_split(train_idx, test_size=0.25, stratify=y[train_idx], random_state=RANDOM_STATE)
mean, sd = float(X[train_idx].mean()), float(X[train_idx].std())


def to_tensor(rows):
    return torch.tensor((X[rows] - mean) / sd, dtype=torch.float32).unsqueeze(1)


Xtr, Xva, Xte = to_tensor(train_idx), to_tensor(val_idx), to_tensor(test_idx)
ytr, yva, yte = (torch.tensor(y[rows]) for rows in (train_idx, val_idx, test_idx))
print(Xtr.shape, Xva.shape, Xte.shape, "| ortalama", round(mean, 4), " SD", round(sd, 4))
print("kümelere göre anormal:", int(ytr.sum()), int(yva.sum()), int(yte.sum()))

**Python notu.** `train_test_split` iki kez çağrılır: Önce yüzde 20'yi test için ayırır, ardından geri kalanın yüzde 25'ini, yani tamamın yüzde 20'sini doğrulama için ayırır. `stratify=y` anormal karelerin oranını her kümede aynı tutar. `.unsqueeze(1)` 1. konuma kanal boyutunu ekler; böylece (n, 64, 64) boyutu (n, 1, 64, 64) olur. Solunda üç ad bulunan satır, parantez içinde yazılmış bir döngü olan *üreteç ifadesinin* oluşturduğu üç tensörü bu adlara dağıtır.

**Alıştırma C.** Test kümesinde kaç anormal kare vardır?

<details><summary><b>Prompt yolu</b>, Alıştırma C: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `X`, sanal kliniğin her hastası için bir diyastol sonu parasternal kısa eksen karesi içeren (2000, 64, 64) boyutunda bir NumPy dizisidir; piksel değerleri 0 ile 1 arasındadır ve bir piksel 2 mm'dir. `y`, eko laboratuvarının raporunu tutar, 0 ya da 1'dir (1 anormal: sol ventrikül diyastol sonu çapı 58 mm'nin üstünde ya da septum 13 mm veya daha kalın). `echo_table`; patient_id, lvedd_mm, ivs_mm ve pw_mm (mm) ile dilated ve thick_septum (0 ya da 1) sütunlarını tutar. `train_idx`, `val_idx` ve `test_idx`, `y`'ye göre tabakalı olarak ayrılmış 1200, 400 ve 400 karenin indisleridir. `Xtr`, `Xva` ve `Xte`, eğitim karelerinin ortalaması ve SS'siyle normalleştirilmiş (n, 1, 64, 64) boyutunda float tensörlerdir; `ytr`, `yva` ve `yte` ise 0 ya da 1 olan etiket tensörleridir.
Görev: Test kümesindeki anormal kareleri (etiket 1) say.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Sayı, tam sayı olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma C", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Eğitim, doğrulama ve test kümeleri toplam 2000 kare etmeli; anormal karelerin oranı üç kümede de yaklaşık aynı olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma C", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = int(yte.sum())
teach.check("Alıştırma C", answer)

**Prompt görevi, 3. bölüm: Yan yana normal ve anormal kareler.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `X`, sanal kliniğin her hastası için bir diyastol sonu parasternal kısa eksen karesi içeren (2000, 64, 64) boyutunda bir NumPy dizisidir; piksel değerleri 0 ile 1 arasındadır ve bir piksel 2 mm'dir. `y`, eko laboratuvarının raporunu tutar, 0 ya da 1'dir (1 anormal: sol ventrikül diyastol sonu çapı 58 mm'nin üstünde ya da septum 13 mm veya daha kalın). `echo_table`; patient_id, lvedd_mm, ivs_mm ve pw_mm (mm) ile dilated ve thick_septum (0 ya da 1) sütunlarını tutar.
Görev: Dört normal ve dört anormal kareyi 2 × 4'lük bir ızgarada, her birinin başlığına lvedd_mm ve ivs_mm değerini yazarak göster.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Gösterilen her karenin etiketini yazdır; ilk satır tamamen 0, ikinci satır tamamen 1 olmalı.
</pre>

</details>

**Beklenen sonuç.** Anormal kareler ya geniş bir boşluk ya da kalın bir duvar gösterir; kalın duvarların bazılarını gözle normalden ayırmak zordur.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 4. Elle ve PyTorch ile evrişim

*Ders: [Evrişim: Görüntü üzerinde kayan filtreler](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#convolution-filters-that-slide-over-an-image)*

Bir evrişim, küçük bir ağırlık tablosunu, yani *çekirdeği*, görüntü üzerinde kaydırır. Her konumda çekirdeği altındaki piksellerle çarpar ve çarpımları toplar; bu, çıktının, yani *öznitelik haritasının*, bir pikselini verir. Aynı çekirdek her yerde kullanıldığı için bir katman az ağırlıkla yetinir ve bir örüntü nerede görünürse görünsün bulunur. *Maksimum havuzlama* her 2 × 2'lik bloğun en büyük değerini tutar ve boyutu yarıya indirir.

**Formüllerden koda.** Çıktı $Y(i, j) = \sum_{u}\sum_{v} K(u, v)\,X(i+u, j+v) + b$ ile hesaplanır: `F.conv2d(x, kernel, bias)`. Girdi $n$ piksel, çekirdek $k$, dolgu $p$ ve adım $s$ olduğunda çıktı $\lfloor (n + 2p - k)/s \rfloor + 1$ pikseldir: `conv_out(n, k, s, p)`. Bu formül aslında bir çapraz korelasyondur; ağ çekirdeği her iki durumda da öğrendiği için derin öğrenmede evrişim adı kullanılır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 4. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(4)

In [ ]:
# 5 × 5'lik bir yamada 3 × 3'lük bir çekirdek, elle ve PyTorch ile
patch = np.array([[0, 0, 0, 0, 0],
                  [0, 1, 1, 1, 0],
                  [0, 1, 0, 1, 0],
                  [0, 1, 1, 1, 0],
                  [0, 0, 0, 0, 0]], dtype=float)
kernel = np.array([[-1, 0, 1],
                   [-1, 0, 1],
                   [-1, 0, 1]], dtype=float)          # dikey kenarlar
by_hand = np.zeros((3, 3))
for r in range(3):
    for c in range(3):
        by_hand[r, c] = np.sum(patch[r:r + 3, c:c + 3] * kernel)
print(by_hand)
with_torch = F.conv2d(torch.tensor(patch)[None, None], torch.tensor(kernel)[None, None])
print(with_torch[0, 0].numpy())


def conv_out(n, k, s=1, p=0):
    return (n + 2 * p - k) // s + 1


for n, k, s, p in [(64, 3, 1, 1), (64, 3, 1, 0), (64, 5, 2, 2), (32, 2, 2, 0)]:
    print(f"n={n} k={k} s={s} p={p} -> {conv_out(n, k, s, p)}")

**Python notu.** İç içe iki `for` döngüsü 3 × 3'lük çıktının her konumunu dolaşır; `patch[r:r + 3, c:c + 3]` o konumdaki 3 × 3'lük penceredir. `[None, None]` öne iki boyut ekler, çünkü `F.conv2d` görüntü için (sayı, kanal, satır, sütun), çekirdek için (çıkış kanalı, giriş kanalı, satır, sütun) boyutunu bekler. `//` bölme yapar ve kalanı atar.

In [ ]:
# VC-1873'ün, yani dilate bir ventrikülün karesinde üç çekirdek
frame_d = plt.imread(ROOT / "data" / "examples" / "echo" / "VC-1873.png")
kernels = {"dikey kenarlar": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
           "yatay kenarlar": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
           "bulanıklaştırma": [[1 / 9, 1 / 9, 1 / 9]] * 3}
t = torch.tensor(frame_d)[None, None]
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(frame_d, cmap="gray")
axes[0].set_title("VC-1873")
for ax, (name, k) in zip(axes[1:], kernels.items()):
    fmap = F.conv2d(t, torch.tensor(k, dtype=torch.float32)[None, None], padding=1)[0, 0]
    ax.imshow(fmap.numpy(), cmap="RdBu_r")
    ax.set_title(name)
for ax in axes:
    ax.axis("off")
plt.show()
print(tuple(t.shape), "→ max pooling →", tuple(F.max_pool2d(t, 2).shape))

**Python notu.** Bir sözlük adları ve değerleri çiftler hâlinde tutar; `.items()` çiftleri döngü için tek tek döndürür. `padding=1` görüntünün çevresine sıfırlardan bir çerçeve ekler; böylece 3 × 3'lük bir çekirdek boyutu 64 × 64'te tutar. `RdBu_r` renk haritası negatif değerleri mavi, pozitif değerleri kırmızı gösterir; bir kenar filtresi bir duvarın iki yanında böyle cevap verir.

**Alıştırma D.** 5 × 5 çekirdekli, adımı 2 ve dolgusu 1 olan bir evrişim 64 × 64 piksellik bir girdi üzerinde çalışmaktadır. Çıktının bir kenarının boyutu nedir?

<details><summary><b>Prompt yolu</b>, Alıştırma D: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: Tek kanallı, 64 × 64 piksellik kare bir girdi.
Görev: 5 × 5 çekirdekli, adımı (stride) 2 ve dolgusu (padding) 1 olan bir evrişimin çıktı boyutunu floor((n + 2p − k) / s) + 1 formülüyle hesapla ve `torch.zeros(1, 1, 64, 64)` üzerinde `F.conv2d` ile doğrula.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Çıktı boyutu (bir kenar), tam sayı olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma D", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 3 × 3 çekirdek, 1 adım ve 1 dolguyla boyut 64 kalmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma D", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = conv_out(64, 5, 2, 1)
print(F.conv2d(torch.zeros(1, 1, 64, 64), torch.zeros(1, 1, 5, 5), stride=2, padding=1).shape)
teach.check("Alıştırma D", answer)

**Prompt görevi, 4. bölüm: Dilate bir ventrikülde iki filtre.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `frame_d`, VC-1873'ün karesidir; değerleri 0 ile 1 arasında olan (64, 64) boyutunda bir NumPy dizisi.
Görev: 3 × 3'lük bir ortalama çekirdeği (bütün değerler 1/9) ile 3 × 3'lük Laplace çekirdeğini [[0, 1, 0], [1, −4, 1], [0, 1, 0]] `F.conv2d` ve 1 dolguyla uygula; kareyi ve iki sonucu yan yana göster.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Üç panelli bir şekil. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: İki sonuç da (64, 64) boyutunda olmalı.
</pre>

</details>

**Beklenen sonuç.** Ortalama çekirdeği benek gürültüsünü yumuşatır; Laplace çekirdeği miyokardın sınırlarını belirginleştirir.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 5. İlk evrişimli ağ

*Ders: [Eko laboratuvarı için bir evrişimli ağ](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#a-convolutional-network-for-the-echo-laboratory)*

Eko laboratuvarının ağında üç blok vardır. Her blokta 3 × 3 çekirdekli bir evrişim, yığın normalizasyonu, ReLU ve maksimum havuzlama bulunur; böylece kare 64 pikselden 32, 16 ve 8 piksele küçülürken öznitelik haritalarının sayısı 8'den 16 ve 32'ye çıkar. Bir doğrusal katman 32 × 8 × 8 = 2048 değeri her sınıf için bir tane olmak üzere iki çıktıya dönüştürür. 2. bölümdeki eğitim döngüsü artık 32 karelik *mini yığınlar* üzerinde çalışır; bütün eğitim karelerinin bir kez geçilmesine bir *dönem (epoch)* denir. Her dönemin sonunda doğrulama karelerindeki kayıp, hangi ağırlıkların saklanacağına karar verir.

**Formüllerden koda.** `EchoCNN` sınıfı katmanları `__init__` içinde sıralar ve bir karenin bunlardan nasıl geçeceğini `forward` içinde söyler. `train_model` dönemleri çalıştırır ve her dönemin eğitim ve doğrulama kaybını döndürür. `F.softmax(model(x), dim=1)[:, 1]` anormal sınıfın olasılığıdır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 5. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(5)

In [ ]:
# Eko laboratuvarının ağı
class EchoCNN(nn.Module):
    """64 × 64 piksellik kareler için üç evrişim bloğu ve bir doğrusal katman."""

    def __init__(self, n_classes=2, dropout=0.3, batchnorm=True):
        super().__init__()
        layers, c_in = [], 1
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


print(EchoCNN())
with torch.no_grad():
    print(EchoCNN()(Xtr[:4]).shape)

**Python notu.** Bir *sınıf* nesneler için bir tariftir. `class EchoCNN(nn.Module)` PyTorch'un `nn.Module` sınıfı üzerine kurulur; `super().__init__()` üst sınıfın önce kendini hazırlamasını sağlar. `self.features` katmanları nesnenin içinde saklar. `layers += [...]` bir listeye ekleme yapar; `[...] if batchnorm else []` yığın normalizasyonunu yalnızca istendiğinde ekler. `nn.Sequential(*layers)` listeyi ayrı argümanlara açar.

**Alıştırma E.** `EchoCNN(n_classes=2)` modelinin kaç eğitilebilir parametresi vardır?

<details><summary><b>Prompt yolu</b>, Alıştırma E: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `EchoCNN`, 5. bölümdeki sınıftır: 8, 16 ve 32 kanallı üç Conv2d (3 × 3, padding 1), BatchNorm2d, ReLU ve MaxPool2d(2) bloğu; ardından Flatten, Dropout(0.3) ve Linear(2048, n_classes).
Görev: `EchoCNN(n_classes=2)` oluştur ve eğitilebilir parametrelerini say.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Parametre sayısı, tam sayı olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma E", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: `nn.Conv2d(1, 8, 3)` katmanının parametrelerini de say; 80 olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma E", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = sum(p.numel() for p in EchoCNN(n_classes=2).parameters() if p.requires_grad)
teach.check("Alıştırma E", answer)

In [ ]:
# Mini yığınlarla eğitim; en iyi dönemin ağırlıkları saklanır
def train_model(model, Xtr, ytr, Xva, yva, epochs=15, lr=3e-3, batch=32, augment_fn=None, seed=RANDOM_STATE):
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history, best_loss, best_state = [], float("inf"), None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(ytr))
        total = 0.0
        for start in range(0, len(ytr), batch):
            j = order[start:start + batch]
            xb = augment_fn(Xtr[j]) if augment_fn else Xtr[j]
            loss = F.cross_entropy(model(xb), ytr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(j)
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(Xva), yva).item()
        history.append((total / len(ytr), val_loss))
        if val_loss < best_loss:
            best_loss, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return history

**Python notu.** `model.train()` ve `model.eval()`, seyreltmeyi ve yığın normalizasyonunu eğitim ve değerlendirme davranışları arasında değiştirir. `torch.randperm` her dönemde karelerin sırasını karıştırır. `copy.deepcopy(model.state_dict())` ağırlıkların tam bir kopyasını saklar; `load_state_dict` sonda en iyi kopyayı geri yükler. Her yığının kaybı büyüklüğüyle, `len(j)` ile, ağırlıklandırılır; böylece son ve daha küçük yığın doğru sayılır.

In [ ]:
# Ağı 15 dönem eğit ve iki kaybı çiz
torch.manual_seed(RANDOM_STATE)
model = EchoCNN()
history = train_model(model, Xtr, ytr, Xva, yva)
best_epoch = int(np.argmin([v for t, v in history])) + 1
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(range(1, len(history) + 1), [t for t, v in history], color="#1F5F8B", marker="o", ms=3, label="eğitim")
ax.plot(range(1, len(history) + 1), [v for t, v in history], color="#9B1B1B", marker="o", ms=3, label="doğrulama")
ax.axvline(best_epoch, color="#888888", ls="--", lw=1)
ax.set_xlabel("dönem")
ax.set_ylabel("kayıp")
ax.legend()
plt.show()
print("en iyi dönem:", best_epoch)

In [ ]:
# Ağ 400 test karesinde
model.eval()
with torch.no_grad():
    p_test = F.softmax(model(Xte), dim=1)[:, 1].numpy()
y_test = yte.numpy()
pred = (p_test >= 0.5).astype(int)
print(confusion_matrix(y_test, pred))
print("doğruluk", round(accuracy_score(y_test, pred), 3),
      "| precision", round(precision_score(y_test, pred), 3),
      "| recall", round(recall_score(y_test, pred), 3),
      "| F1", round(f1_score(y_test, pred), 3),
      "| AUC", round(roc_auc_score(y_test, p_test), 3))

**Gözlemle ve düşün.** Ders aynı ağı aynı tohum değeriyle eğitmiş ve 0,968 doğruluk, 0,822 F1 ve 0,983 alıcı işletim karakteristiği (ROC) eğrisi altındaki alan (AUC) değerine ulaşmıştır. Bilgisayarlar bir evrişimin sayılarını farklı sıralarla topladığı için sizin sayılarınız son basamaklarda farklı olabilir. Eko laboratuvarının sorumlusuna hangi ölçütleri bildirirdiniz ve neden yalnızca doğruluğu değil?

**Prompt görevi, 5. bölüm: Dönem başına doğruluk.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `train_idx`, `val_idx` ve `test_idx`, `y`'ye göre tabakalı olarak ayrılmış 1200, 400 ve 400 karenin indisleridir. `Xtr`, `Xva` ve `Xte`, eğitim karelerinin ortalaması ve SS'siyle normalleştirilmiş (n, 1, 64, 64) boyutunda float tensörlerdir; `ytr`, `yva` ve `yte` ise 0 ya da 1 olan etiket tensörleridir. `EchoCNN`, 5. bölümdeki sınıftır: 8, 16 ve 32 kanallı üç Conv2d (3 × 3, padding 1), BatchNorm2d, ReLU ve MaxPool2d(2) bloğu; ardından Flatten, Dropout(0.3) ve Linear(2048, n_classes).
Görev: Yeni bir `EchoCNN` modelini Adam ile (öğrenme hızı 0,003, 32'lik yığınlar) 10 dönem eğit; her dönemden sonra eğitim ve doğrulama kümesindeki doğruluğu kaydet. İkisini döneme karşı çiz.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: İki çizgili bir şekil. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Doğruluğu değerlendirme kipinde ve `torch.no_grad()` içinde hesapla; iki değer de 0 ile 1 arasında olmalı.
</pre>

</details>

**Beklenen sonuç.** İki eğri de ilk dönemlerden sonra 0,9'u aşar, çünkü karelerin yüzde 90'ı normaldir. 7. bölümdeki F1 değeri daha çok şey söyler.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 6. Aşırı öğrenme: Seyreltme, yığın normalizasyonu ve veri çoğaltma

*Ders: [Aşırı öğrenme: Seyreltme, yığın normalizasyonu ve veri çoğaltma](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#overfitting-dropout-batch-normalisation-and-augmentation)*

Eğitim karesi azsa bir ağ bu kareleri ezberleyebilir: Eğitim kaybı sıfıra doğru inerken doğrulama kaybı düşmeyi bırakır ya da yükselir. Buna karşı üç araç kullanılır. *Seyreltme (dropout)* her eğitim adımında birimlerin rastgele bir $p$ oranını kapatır. *Yığın normalizasyonu* her öznitelik haritasını mini yığının ortalaması ve SS'siyle yeniden ölçekler. *Veri çoğaltma* ağa karelerin hafifçe değiştirilmiş kopyalarını gösterir; burada küçük döndürmeler, kaydırmalar ve parlaklık değişiklikleri kullanılır. *Erken durdurma*, `train_model` fonksiyonunun zaten yaptığı gibi, doğrulama kaybının en düşük olduğu dönemin ağırlıklarını saklar.

**Formüllerden koda.** Eğitimde seyreltme korunan birimleri $1/(1-p)$ ile çarpar; böylece beklenen toplamları değişmez. Değerlendirmede hiçbir şey yapmaz. Yığın normalizasyonu $\hat{x} = (x - \mu_B)/\sqrt{\sigma_B^2 + \varepsilon}$ değerini, ardından da öğrenilen $\gamma$ ve $\beta$ ile $\gamma \hat{x} + \beta$ değerini hesaplar. `augment(x)` her kare için `F.affine_grid` ile bir döndürme ve kaydırma oluşturur ve bunları `F.grid_sample` ile uygular.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 6. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(6)

In [ ]:
# Veri çoğaltma: küçük döndürmeler, kaydırmalar ve parlaklık değişiklikleri
def augment(x, max_deg=15, max_shift=0.08, gain=0.10):
    n = x.shape[0]
    a = (torch.rand(n) * 2 - 1) * math.radians(max_deg)
    s = (torch.rand(n, 2) * 2 - 1) * max_shift
    theta = torch.stack([torch.stack([a.cos(), -a.sin(), s[:, 0]], 1),
                         torch.stack([a.sin(), a.cos(), s[:, 1]], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, padding_mode="border", align_corners=False)
    return x * (1 + (torch.rand(n, 1, 1, 1) * 2 - 1) * gain)


torch.manual_seed(RANDOM_STATE)
case_row = list(echo_table.patient_id).index("VC-1212")
copies = augment(to_tensor([case_row]).repeat(4, 1, 1, 1))
fig, axes = plt.subplots(1, 5, figsize=(11, 2.5))
axes[0].imshow(X[case_row], cmap="gray")
axes[0].set_title("VC-1212")
for k, ax in enumerate(axes[1:]):
    ax.imshow(copies[k, 0].numpy(), cmap="gray")
    ax.set_title(f"kopya {k + 1}")
for ax in axes:
    ax.axis("off")
plt.show()

**Python notu.** `torch.rand(n)` 0 ile 1 arasında n sayı çeker; böylece `torch.rand(n) * 2 - 1` −1 ile 1 arasında kalır. `theta` her kare için 2 × 3'lük bir matris tutar: İlk iki sütun döndürür, sonuncusu kaydırır. `.repeat(4, 1, 1, 1)` kareyi ilk boyut boyunca dört kez kopyalar. Veri çoğaltma kareyi soldan sağa çevirmez: Bu, sağ ventrikülü kalbin yanlış tarafına koyardı.

In [ ]:
# 300 eğitim karesi, 40 dönem: veri çoğaltmasız ve veri çoğaltmalı
torch.manual_seed(RANDOM_STATE)
plain = EchoCNN(dropout=0.0, batchnorm=False)
hist_plain = train_model(plain, Xtr[:300], ytr[:300], Xva, yva, epochs=40)
torch.manual_seed(RANDOM_STATE)
augmented = EchoCNN(dropout=0.0, batchnorm=False)
hist_aug = train_model(augmented, Xtr[:300], ytr[:300], Xva, yva, epochs=40, augment_fn=augment)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, h, name in [(axes[0], hist_plain, "düzenlileştirme yok"), (axes[1], hist_aug, "veri çoğaltma")]:
    ax.plot(range(1, 41), [t for t, v in h], color="#1F5F8B", label="eğitim")
    ax.plot(range(1, 41), [v for t, v in h], color="#9B1B1B", label="doğrulama")
    ax.set_title(name)
    ax.set_xlabel("dönem")
    ax.set_yscale("log")
axes[0].set_ylabel("kayıp (logaritmik ölçek)")
axes[1].legend()
plt.show()
for name, h in [("plain", hist_plain), ("augmented", hist_aug)]:
    print(name, "son dönem: eğitim", round(h[-1][0], 4), "doğrulama", round(h[-1][1], 4))

**Gözlemle ve düşün.** Düzenlileştirme olmadan eğitim kaybı doğrulama kaybının çok altına iner: Ağ 300 kareyi ezberlemiştir. Veri çoğaltmayla iki eğri birbirine yakın kalır. Bir karenin döndürülmüş kopyası neden ağ için yeni bilgi sayılır da bir kardiyolog için sayılmaz?

In [ ]:
# Küçük tensörlerde seyreltme ve yığın normalizasyonu
torch.manual_seed(RANDOM_STATE)
ones = torch.ones(10)
print("seyreltme, eğitim:  ", F.dropout(ones, p=0.3, training=True).numpy().round(4))
print("seyreltme, değerlendirme:", F.dropout(ones, p=0.3, training=False).numpy())
bn = nn.BatchNorm1d(1)
batch_of_four = torch.tensor([[2.0], [4.0], [6.0], [8.0]])
print("2, 4, 6, 8 değerlerinin yığın normalizasyonu:", bn(batch_of_four).detach().flatten().numpy().round(3))

**Alıştırma F.** Eğitim kipinde p = 0,3 olan seyreltme, değerlerin ortalama yüzde 30'unu 0 yapar. `torch.ones(10)` tensörünün korunan elemanları hangi değeri alır? Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma F: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: On tane birden oluşan bir tensör, `torch.ones(10)`.
Görev: `F.dropout` fonksiyonunu eğitim kipinde p = 0,3 ile uygula ve korunan elemanların aldığı değeri bul.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Korunan bir elemanın değeri, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma F", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: `training=False` ile tensör değişmeden geri dönmeli.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma F", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
kept = F.dropout(torch.ones(10), p=0.3, training=True)
answer = round(float(kept[kept > 0][0]), 3)
teach.check("Alıştırma F", answer)

**Prompt görevi, 6. bölüm: Vakanın sekiz çoğaltılmış kopyası.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `frame`, VC-1212'nin karesidir, (64, 64) boyutunda bir NumPy dizisi; `augment(x)` ise 6. bölümdeki, (n, 1, 64, 64) boyutunda bir tensör alan fonksiyondur.
Görev: `augment` ile karenin sekiz çoğaltılmış kopyasını oluştur ve 2 × 4'lük bir ızgarada göster.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Her kopya (64, 64) boyutunu korumalı ve değerleri karenin değer aralığına yakın kalmalı.
</pre>

</details>

**Beklenen sonuç.** Kalp birkaç piksel döner ve kayar, parlaklık değişir; anatomi ise akla yatkın kalır. Sağ-sol çevirme sağ ventrikülü yanlış tarafa taşırdı.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 7. Sonucu okumak: Kesinlik, duyarlılık ve F1

*Ders: [Sonucu okumak: Kesinlik, duyarlılık ve F1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#reading-the-result-precision-recall-and-f1)*

On karenin dokuzu normal olduğunda, her kareye normal diyen bir ağ 0,90 doğruluğa ulaşır ve hiçbir hastalık bulamaz. Bir sınıfa ait ölçütler daha iyi cevap verir: *Kesinlik (precision)* bütün alarmlar arasında doğru alarmların oranı, *duyarlılık (recall)* hastalıklı karelerin bulunan oranı, *F1 skoru* ise bu ikisinin harmonik ortalamasıdır. Birden çok sınıfta *makro F1*, sınıfların F1 değerlerinin düz ortalamasıdır; böylece küçük bir sınıf büyük bir sınıf kadar sayılır. İkinci hücre, sine kardiyak manyetik rezonansın (KMR) 180 kısa eksen kesitinde üç sınıflı bir ağın sonucunu okur: normal, dilate kardiyomiyopati (DCM) ve hipertrofik kardiyomiyopati (HCM).

**Formüllerden koda.** $k$ sınıfı için $P_k = TP_k/(TP_k + FP_k)$, $R_k = TP_k/(TP_k + FN_k)$ ve $F1_k = 2 P_k R_k/(P_k + R_k)$'dir. Satırlarda gerçek sınıfın bulunduğu bir `cm` karışıklık matrisinde `np.diag(cm)` $TP_k$ değerlerini tutar; sütun toplamları `cm.sum(axis=0)` $TP_k + FP_k$, satır toplamları `cm.sum(axis=1)` ise $TP_k + FN_k$ değerini verir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 7. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(7)

In [ ]:
# Ağ hangi anormal kareleri kaçırıyor?
test_table = echo_table.iloc[test_idx].reset_index(drop=True)
test_table["p_abnormal"] = p_test.round(3)
test_table["pred"] = pred
dilated = test_table["dilated"] == 1
thick_only = (test_table["thick_septum"] == 1) & ~dilated
print("dilate ventrikülde duyarlılık:", round(test_table.loc[dilated, "pred"].mean(), 3), f"(n={dilated.sum()})")
print("yalnızca kalın septumda duyarlılık:", round(test_table.loc[thick_only, "pred"].mean(), 3), f"(n={thick_only.sum()})")
test_table.loc[(test_table["label"] == 1) & (test_table["pred"] == 0),
               ["patient_id", "lvedd_mm", "ivs_mm", "dilated", "thick_septum", "p_abnormal"]]

**Python notu.** `echo_table.iloc[test_idx]` test karelerinin satırlarını konumlarına göre seçer; `.reset_index(drop=True)` onları `p_test` ile aynı sırada, 0'dan başlayarak yeniden numaralar. `~` True'yu False'a, False'u True'ya çevirir; `&` iki koşulu birleştirir. `.loc[rows, columns]` satırları bir koşula, sütunları ada göre seçer.

In [ ]:
# Elle üç sınıf: Bir ağın 180 KMR kesitindeki sonucu
cm3 = np.array([[51, 8, 1],
                [0, 60, 0],
                [6, 0, 54]])
tp = np.diag(cm3)
precision = tp / cm3.sum(axis=0)
recall = tp / cm3.sum(axis=1)
f1 = 2 * precision * recall / (precision + recall)
pd.DataFrame({"precision": precision, "recall": recall, "F1": f1}, index=CMR_CLASSES).round(3)

**Alıştırma G.** `cm3` matrisindeki üç sınıfın makro F1 değeri nedir? Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma G: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: Bir ağın 180 kısa eksen KMR kesitindeki karışıklık matrisi; satırlar gerçek, sütunlar tahmin edilen sınıf, sıra normal, DCM ve HCM: [[51, 8, 1], [0, 60, 0], [6, 0, 54]].
Görev: Matristen her sınıfın kesinliğini, duyarlılığını ve F1 değerini, ardından bunların ortalaması olan makro F1'i hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Makro F1, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma G", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: DCM'nin F1 değeri 0,938 olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma G", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(float(f1.mean()), 3)
teach.check("Alıştırma G", answer)

**Gözlemle ve düşün.** Ağ dilate ventriküllerin çoğunu bulur, ancak tek bulgusu 13-14 mm'lik bir septum olan karelerin çoğunu kaçırır. Piksel başına 2 mm'de normal 10 mm'lik bir septumla fark bir ya da iki pikseldir ve eğitim kümesinde böyle yalnızca otuz kadar kare vardır. İlk önce neyi değiştirirdiniz: çözünürlüğü, örnek sayısını mı yoksa eşiği mi?

**Prompt görevi, 7. bölüm: Eşiklere göre kesinlik ve duyarlılık.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `p_test`, 400 test karesi için anormal sınıfın olasılıkları; `y_test`, etiketleri (0 ya da 1).
Görev: 0,1, 0,2, …, 0,9 eşiklerinin her biri için anormal sınıfın kesinliğini ve duyarlılığını hesapla; ikisini eşiğe karşı çiz.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil ve dokuz eşiğin tablosu. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 0,5 eşiğinde tablo, üstteki değerlendirmenin değerlerini göstermeli.
</pre>

</details>

**Beklenen sonuç.** Eşik büyüdükçe duyarlılık düşer, kesinlik yükselir; daha düşük bir eşik, daha fazla yanlış alarm pahasına daha çok kalın septum bulur.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 8. Bilinen mimariler ve atlama bağlantıları

*Ders: [Bilinen mimariler: LeNet'ten ResNet'e](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#known-architectures-from-lenet-to-resnet)*

Evrişimli ağlar, el yazısı rakamları yaklaşık altmış bin ağırlıkla okuyan LeNet-5'ten milyonlarca ağırlığı ve yüzden fazla katmanı olan ağlara kadar büyümüştür. Çok derin ağlar ResNet'in *atlama bağlantısıyla* eğitilebilir hâle gelmiştir: Bir blok girdisini çıktısına ekler, $y = F(x) + x$; böylece gradyan bloğun yanından akabilir ve ekleyecek bir şeyi olmayan bir blok $F(x) = 0$ öğrenebilir.

**Formüllerden koda.** $c_{in}$ giriş kanalı, $c_{out}$ çıkış kanalı ve $k \times k$ çekirdeği olan bir evrişim katmanında $c_{in} \cdot c_{out} \cdot k^2 + c_{out}$ parametre bulunur; son terim sabit terimler içindir. Bir yığın normalizasyonu katmanında $2c$ parametre vardır: $c$ kanalın her biri için $\gamma$ ve $\beta$. `count(model)` eğitilebilir bütün tensörlerin boyutlarını toplar.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 8. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(8)

In [ ]:
# LeNet-5'in, bir VGG bloğunun ve bir artık bloğun parametreleri
def count(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


lenet5 = nn.Sequential(nn.Conv2d(1, 6, 5), nn.Tanh(), nn.AvgPool2d(2),
                       nn.Conv2d(6, 16, 5), nn.Tanh(), nn.AvgPool2d(2), nn.Flatten(),
                       nn.Linear(16 * 5 * 5, 120), nn.Tanh(), nn.Linear(120, 84), nn.Tanh(), nn.Linear(84, 10))
vgg_block = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
                          nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))


class BasicBlock(nn.Module):
    """ResNet'in temel bloğu: iki 3 × 3 evrişim ve bir atlama bağlantısı."""

    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)            # atlama bağlantısı


print("LeNet-5 (32 × 32):", count(lenet5), "|", tuple(lenet5(torch.zeros(1, 1, 32, 32)).shape))
print("VGG bloğu 64 → 128:", count(vgg_block))
print("EchoCNN:", count(EchoCNN()))
try:
    import torchvision
    print("ResNet-18 (torchvision):", count(torchvision.models.resnet18(weights=None)))
except ImportError:
    print("torchvision kurulu değil; ResNet-18'in 11.689.512 parametresi vardır.")

**Python notu.** `bias=False` evrişimlerin sabit terimlerini çıkarır, çünkü hemen ardından gelen yığın normalizasyonunun kendi kaydırma terimi $\beta$ vardır. `forward` içindeki `out + x` atlama bağlantısıdır; `out` ile `x`'in aynı boyutta olmasını gerektirir ve 1'lik dolgu bunu sağlar. `weights=None`, ResNet-18'i rastgele ağırlıklarla kurar; böylece hiçbir şey indirilmez.

**Alıştırma H.** 16 giriş kanalı, 32 çıkış kanalı, 3 × 3 çekirdeği ve sabit terimleri olan bir evrişim katmanının kaç parametresi vardır?

<details><summary><b>Prompt yolu</b>, Alıştırma H: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: 16 giriş kanalı, 32 çıkış kanalı, 3 × 3 çekirdeği ve her çıkış kanalı için bir sabit terimi olan bir evrişim katmanı.
Görev: Parametrelerini c_in × c_out × k × k + c_out formülüyle say ve sonucu `nn.Conv2d` ile doğrula.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Parametre sayısı, tam sayı olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma H", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: `nn.Conv2d(1, 8, 3)` katmanının 80 parametresi olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma H", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = count(nn.Conv2d(16, 32, 3))
teach.check("Alıştırma H", answer)

**Alıştırma I.** `BasicBlock(64)` bloğunun kaç eğitilebilir parametresi vardır?

<details><summary><b>Prompt yolu</b>, Alıştırma I: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: 64 kanallı bir ResNet temel bloğu: dolgusu 1 olan ve sabit terim içermeyen iki 3 × 3 evrişim, her birinin ardından `nn.BatchNorm2d(64)`, ilkinin ardından ReLU ve son ReLU'dan önce girdinin çıktıya eklenmesi.
Görev: Bloğu `nn.Module` alt sınıfı olarak yaz ve eğitilebilir parametrelerini say.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Parametre sayısı, tam sayı olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma I", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Bloğun `torch.randn(1, 64, 16, 16)` için verdiği çıktı, girdiyle aynı boyutta olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma I", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = count(BasicBlock(64))
print(BasicBlock(64)(torch.randn(1, 64, 16, 16)).shape)
teach.check("Alıştırma I", answer)

**Prompt görevi, 8. bölüm: Katman katman parametreler.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `EchoCNN`, 5. bölümdeki sınıftır: 8, 16 ve 32 kanallı üç Conv2d (3 × 3, padding 1), BatchNorm2d, ReLU ve MaxPool2d(2) bloğu; ardından Flatten, Dropout(0.3) ve Linear(2048, n_classes).
Görev: `EchoCNN(n_classes=2)` modelinin parametresi olan her katmanı için bir satır içeren bir tablo oluştur: katmanın adı, türü ve parametre sayısı.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir pandas DataFrame'i. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Parametre sütununun toplamı Alıştırma E'deki toplama eşit olmalı.
</pre>

</details>

**Beklenen sonuç.** 10098 parametrenin 4098'i doğrusal katmandadır; üç evrişim katmanı toplam 5888 parametre tutar.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 9. YZ ile eşli kodlama, bölüm 3: Bir eğitim döngüsünü denetlemek

*Ders: [YZ ile eşli kodlama, bölüm 3: Bir eğitim döngüsünü denetlemek](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#ai-pair-programming-part-3-reviewing-a-training-loop)*

Bir katılımcı bir YZ asistanından *dersin ağını test karelerinde değerlendirmesini* istemiştir. Aşağıdaki kod asistandan dönen cevaptır. Kod çalışır ve yüksek bir doğruluk bildirir. Sayıya güvenmeden önce kodu dersteki eğitim döngüsü denetiminin beş sorusuyla okuyun: Normalleştirmenin istatistikleri nereden gelmektedir? Model değerlendirme kipinde midir ve gradyanlar kapalı mıdır? Çıktı katmanı kayıp fonksiyonuna uyuyor mu? Nadir bir sınıfa hangi ölçüt uyar?

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 9. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(9)

In [ ]:
# Bir YZ asistanının kodu: dersin ağını değerlendir
course_model = EchoCNN()
course_model.load_state_dict(torch.load(ROOT / "models" / "week03" / "echo_cnn.pt", weights_only=True))
X_all = torch.tensor((X - X.mean()) / X.std(), dtype=torch.float32).unsqueeze(1)
probs = F.softmax(course_model(X_all[test_idx]), dim=1)
accuracy = (probs.argmax(dim=1).numpy() == y[test_idx]).mean()
print("Accuracy:", round(float(accuracy), 3))

**Gözlemle ve düşün.** Kodun üç sorunu ve bir eksiği vardır. Normalleştirmenin istatistikleri 2000 karenin tamamından gelmektedir. Model eğitim kipinde bırakılmıştır; bu yüzden seyreltme birimleri rastgele kapatır, yığın normalizasyonu test karelerinin istatistiklerini kullanır ve autograd her işlemin kaydını tutar. Tek başına doğruluk, her dört anormal kareden birinin kaçırıldığını gizler. Hücreyi iki kez çalıştırın: Doğruluk neden değişiyor?

**Alıştırma J.** `course_model` modelini doğru değerlendirin: Değerlendirme kipine alın, `torch.no_grad()` içinde `Xte` dizisindeki her test karesinin sınıfını tahmin edin ve `yte` ile karşılaştırarak anormal sınıfın F1 değerini hesaplayın. Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma J: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `course_model`, models/week03/echo_cnn.pt dosyasından yüklenmiş, dersin ağırlıklarını taşıyan bir `EchoCNN` nesnesidir. `Xte` ve `yte`, 400 test karesi ve etiketleridir (0 ya da 1, 40'ı anormal).
Görev: Modeli değerlendirme kipine al, `torch.no_grad()` içinde her test karesi için en büyük çıktıya sahip sınıfı tahmin et ve anormal sınıfın F1 değerini `f1_score` ile hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: F1, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma J", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Doğruluğu da yazdır: 0,95'in üstünde olmalı, F1 ise belirgin biçimde daha düşük kalmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma J", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
course_model.eval()
with torch.no_grad():
    pred_course = course_model(Xte).argmax(dim=1).numpy()
print("accuracy", round(accuracy_score(yte.numpy(), pred_course), 3))
answer = round(f1_score(yte.numpy(), pred_course), 3)
teach.check("Alıştırma J", answer)

In [ ]:
# Haftanın vakası VC-1212, dersin ağından geçerken
course_model.eval()
with torch.no_grad():
    p_case = F.softmax(course_model(to_tensor([case_row])), dim=1)[0, 1].item()
print("VC-1212: anormal kare olasılığı", round(p_case, 3),
      "| lvedd", echo_table.lvedd_mm[case_row], "mm | ivs", echo_table.ivs_mm[case_row], "mm")

Etkileşimli labdaki [prompt atölyesi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/lab.html#part-f) böyle bir değerlendirmenin promptunu prompt kartının yedi alanıyla yazar. *Test* alanı, denetimin bulduğu sorunları yakalayacak kontrolü tutar: İki kez çalıştırılan aynı değerlendirme aynı sayıları vermelidir.

**Prompt görevi, 9. bölüm: Bir modeli değerlendiren fonksiyon.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt adıyla; PyTorch torch, torch.nn nn ve torch.nn.functional F adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi, teach ve imaging modülleriyle birlikte kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: Dersin `EchoCNN` modeli olan `course_model` ile `Xte` ve `yte` test tensörleri.
Görev: Modeli değerlendirme kipine alan, `torch.no_grad()` içinde tahmin yapan ve anormal sınıfın doğruluk, kesinlik, duyarlılık ve F1 değerlerini içeren bir sözlük döndüren `evaluate(model, X, y)` fonksiyonunu yaz.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Fonksiyon ve ders modeli için sonucu, üç ondalığa yuvarlanmış olarak. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Fonksiyon iki kez çağrıldığında aynı sayıları vermeli.
</pre>

</details>

**Beklenen sonuç.** Ders modeli için: doğruluk 0,968, kesinlik 0,909, duyarlılık 0,75 ve F1 0,822.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 10. Uygulama anahtarı: Başka bir veri seti, başka ayarlar

`DATASET`, `EPOCHS`, `AUGMENT` ve `DROPOUT` değerlerini değiştirin ve hücreyi yeniden çalıştırın. `DATASET = "cmr"` olduğunda hücre ağı kardiyomiyopati kliniğinin kısa eksen kesitleriyle üç sınıf üzerinde eğitir: normal, DCM ve HCM. Test kümesinin karışıklık matrisini, her sınıfın F1 değerini ve makro F1'i yazdırır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 10. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(10)

In [ ]:
DATASET = "cmr"        # "echo" ya da "cmr"
EPOCHS = 15
AUGMENT = True
DROPOUT = 0.3

if DATASET == "cmr":
    Xd, yd, table_d = make_cmr_set(300)
    names = CMR_CLASSES
else:
    Xd, yd, table_d = X, y, echo_table
    names = ECHO_CLASSES
rows = np.arange(len(yd))
tr_d, te_d = train_test_split(rows, test_size=0.2, stratify=yd, random_state=RANDOM_STATE)
tr_d, va_d = train_test_split(tr_d, test_size=0.25, stratify=yd[tr_d], random_state=RANDOM_STATE)
mu, sigma = Xd[tr_d].mean(), Xd[tr_d].std()
T = lambda r: torch.tensor((Xd[r] - mu) / sigma, dtype=torch.float32).unsqueeze(1)
torch.manual_seed(RANDOM_STATE)
net = EchoCNN(n_classes=len(names), dropout=DROPOUT)
train_model(net, T(tr_d), torch.tensor(yd[tr_d]), T(va_d), torch.tensor(yd[va_d]), epochs=EPOCHS,
            augment_fn=augment if AUGMENT else None)
net.eval()
with torch.no_grad():
    pred_d = net(T(te_d)).argmax(dim=1).numpy()
print(pd.DataFrame(confusion_matrix(yd[te_d], pred_d), index=names, columns=names))
print("F1:", dict(zip(names, f1_score(yd[te_d], pred_d, average=None).round(3))))
print("makro F1:", round(f1_score(yd[te_d], pred_d, average="macro"), 3))

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Kalp yetersizliği | Eko karelerine `echo_table` tablosundan iki yerine üç sınıf verin: normal, dilate ve kalın septum; ardından `EchoCNN(n_classes=3)` modelini eğitin. Ağ en sık hangi sınıfı karıştırıyor ve neyle? |
| Girişimsel kardiyoloji | Ağı eğitim karelerinin yüzde 25'i, 50'si ve 100'üyle eğitin; anormal sınıfın F1 değerini eğitim kümesinin büyüklüğüne karşı çizin. Yeni bir kateter laboratuvarı, böyle bir ağı eğitmeye değmeden önce kaç örnek toplamalıdır? |
| Elektrofizyoloji | `test_table` tablosunu kayıttaki `ecg_rhythm` sütunuyla birleştirin; atriyal fibrilasyonlu hastalarda ve sinüs ritmindeki hastalarda kaçırılan anormal karelerin oranını karşılaştırın. Tek karelerde bir fark bekler miydiniz, videolarda? |
| Kardiyovasküler görüntüleme | Anormal sınıf olasılığı en yüksek olan on normal test karesini gösterin. Ortak yönleri nedir ve her vakayı hangi ekokardiyografik ölçüm çözerdi? |
| Koruyucu kardiyoloji | Hipertansiyonu olan hastalarda ağ anormal karelerin ne kadarını kaçırmaktadır ve bu oran hipertansiyonu olmayanlarla nasıl karşılaştırılır? Bu sonuç, hipertansif kalp hastalığı için bir tarama programında ne anlama gelir? |
| Kardiyo-onkoloji | Kemoterapi sırasında bir hastayı izleyin: Çap 52 mm'den 64 mm'ye 2 mm'lik adımlarla büyürken VC-0821'in karelerini `vclinic.imaging.echo_psax` ile çizin ve ders ağının anormal sınıf olasılığını çapa karşı çizin. Ağ kararını nerede değiştiriyor? |
| Yoğun bakım | 400 test karesinin tahmin süresini kendi bilgisayarınızda ölçün ve saniyedeki kare sayısını hesaplayın. Ağ, saniyede 50 kare üreten bir yatak başı ekokardiyografisine yetişebilir mi? |

## Defterden sonra

[Etkileşimli labı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/lab.html) açın ve *Kendini sına* sekmesindeki öz değerlendirmeyi yapın. Ardından *Düşün ve dışa aktar* sekmesinde yansıtma metnini yazın ve öğrenme günlüğünü dışa aktarın. [Hafta sayfasında](README.md) anlatılan 3. haftanın haftalık projesi, 5. bölümdeki ağı 6. ve 7. bölümlerin araçlarıyla kardiyomiyopati kliniğinin kesitleri üzerinde eğitir.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>